In [14]:
import cassiopeia as cass
import requests
RIOT_API_KEY = os.environ.get("RIOT_API_KEY", "")
HEADERS = {"X-Riot-Token": RIOT_API_KEY}

In [12]:
REGION_ROUTING = "europe"     
REGION_PLATFORM = "euw1"    

# 1. Summoner Name → Summoner Info
def get_summoner_by_name(name):
    url = f"https://{REGION_PLATFORM}.api.riotgames.com/lol/summoner/v4/summoners/by-name/{name}"
    res = requests.get(url, headers=HEADERS)
    return res.json()

# 2. PUUID → Match-IDs (bis 1000)
def get_match_ids(puuid, count=100, start=0):
    url = f"https://{REGION_ROUTING}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    params = {"start": start, "count": count, "queue": 420}  # Ranked Solo/Duo
    res = requests.get(url, headers=HEADERS, params=params)
    return res.json()

# 3. Match-ID → Match Info
def get_match(match_id):
    url = f"https://{REGION_ROUTING}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    res = requests.get(url, headers=HEADERS)
    return res.json()

# 4. Match-ID → Timeline
def get_timeline(match_id):
    url = f"https://{REGION_ROUTING}.api.riotgames.com/lol/match/v5/matches/{match_id}/timeline"
    res = requests.get(url, headers=HEADERS)
    return res.json()

In [24]:
from riotwatcher import LolWatcher, RiotWatcher, ApiError
my_platform = 'euw1'      # für Summoner-, Ranked-, Champion-APIs
my_routing = 'europe'     # für Match-V5 (Match-IDs, Timelines)

In [30]:
lol_watcher = LolWatcher(RIOT_API_KEY)
riot_watcher = RiotWatcher(RIOT_API_KEY)

riot_account = riot_watcher.account.by_riot_id(my_routing, 'Wollknäul Socken', 'Bart')
summoner = lol_watcher.summoner.by_puuid(my_platform, riot_account['puuid'])

In [31]:
ranked_stats = lol_watcher.league.by_puuid(my_platform, summoner['puuid'])
for queue in ranked_stats:
    print(f"{queue['queueType']}: {queue['tier']} {queue['rank']}, LP: {queue['leaguePoints']}")

match_ids = lol_watcher.match.matchlist_by_puuid(my_routing, summoner['puuid'], count=10)

for match_id in match_ids:
    match = lol_watcher.match.by_id(my_routing, match_id)
    timeline = lol_watcher.match.timeline_by_match(my_routing, match_id)

    print(f"{match_id} - Game Duration: {match['info']['gameDuration']}s")


RANKED_SOLO_5x5: EMERALD II, LP: 66
RANKED_FLEX_SR: PLATINUM III, LP: 73
EUW1_7472556656 - Game Duration: 2007s
EUW1_7472512187 - Game Duration: 1851s
EUW1_7471669174 - Game Duration: 1611s
EUW1_7471626566 - Game Duration: 1143s
EUW1_7471546078 - Game Duration: 1658s
EUW1_7470833780 - Game Duration: 1686s
EUW1_7470786940 - Game Duration: 1681s
EUW1_7470644377 - Game Duration: 1950s
EUW1_7469778861 - Game Duration: 2114s
EUW1_7469705535 - Game Duration: 1975s
